# Les inn Lånekassen-data fra Omsetjingsminne 
Se [https://www.nb.no/sprakbanken/ressurskatalog/oai-nb-no-sbr-78/](https://www.nb.no/sprakbanken/ressurskatalog/oai-nb-no-sbr-78/)

In [42]:
from collections import defaultdict
import pandas as pd
import xml.etree.ElementTree as ET

tree = ET.parse("lanekassen.no.csv-nb-nn.deduped.tmx")
root = tree.getroot()

data_df = defaultdict(list)

body = root.find("body")
tus = body.findall("tu")
for tu in tus:
    if tu.attrib["datatype"] == "Text":
        for e in tu:
            if e.tag == "prop":
                if "score" in e.attrib["type"]:
                    data_df["_".join(e.attrib["type"].split("-"))].append(e.text)
            elif e.tag == "tuv":
                assert len(e.attrib) == 1
                lang = e.attrib["{http://www.w3.org/XML/1998/namespace}lang"]

                urls = [x.text for x in e if x.tag == "prop"]
                texts = [x.text for x in e if x.tag == "seg"]
                assert len(texts) == 1

                data_df[f"{lang}_segment"].append(texts[0])
                data_df[f"{lang}_urls"].append(urls)

df = pd.DataFrame(data_df)
df

,score_hunalign,score_bicleaner,nb_segment,nb_urls,nn_segment,nn_urls
0,2.00697,0.516,Les mer om informasjonssikkerhet i Lånekassen,[http://lanekassen.no/nb-NO/regelverk/andre-re...,Les meir om informasjonstryggleik i Lånekassen,[http://lanekassen.no/nn-NO/regelverk/andre-re...
1,2.16402,0.512,Lånekassen vil da sende deg et brev med nærmer...,[http://lanekassen.no/nb-NO/presse-og-samfunns...,Lånekassen vil då sende deg eit brev med nærar...,[http://lanekassen.no/nn-NO/presse-og-samfunns...
2,2.44286,0.523,Hva betyr sletting av renter?,[http://lanekassen.no/globalassets/skjemaer-fe...,Kva betyr sletting av renter?,[http://lanekassen.no/globalassets/skjemaer-fe...
3,2.62564,0.515,Vi er i ferd med erstatte hele lanekassen.no.,[https://larestedsinfo.lanekassen.no/nb-NO/for...,Vi er i ferd med erstatte heile lanekassen.no.,[https://larestedsinfo.lanekassen.no/nn-NO/reg...
4,3.15714,0.508,For å få full omgjøring må du bestå like mange...,[http://lanekassen.no/nb-NO/stipend-og-lan/omg...,For å få full omgjering må du bestå like mange...,[http://lanekassen.no/nn-NO/stipend-og-lan/omg...
...,...,...,...,...,...,...
2273,2.72455,0.517,Du kan klage på det som kalles et enkeltvedtak.,[http://lanekassen.no/nb-NO/regelverk/klage/],Du kan klage på det som blir kalla eit enkeltv...,[http://lanekassen.no/nn-NO/regelverk/klage/]
2274,2.05652,0.540,Lånekassen har et personvernombud som ivaretar...,[http://lanekassen.no/nb-NO/regelverk/andre-re...,Lånekassen har eit personvernombod som ivarete...,[http://lanekassen.no/nn-NO/regelverk/andre-re...
2275,2.66437,0.500,Det er noen generelle vilkår du må oppfylle fo...,[http://lanekassen.no/nb-NO/regelverk/tildelin...,Det er somme generelle vilkår du må fylle for ...,[http://lanekassen.no/nn-NO/regelverk/tildelin...
2276,2.13045,0.568,fulltidsarbeid i minst seks måneder av opptjen...,[http://lanekassen.no/nb-NO/gjeld-og-betaling/...,fulltidsarbeid i minst seks månader av oppteni...,[http://lanekassen.no/nn-NO/gjeld-og-betaling/...


# Match opp til doc_hash via url

In [7]:
lånekassen_data = pd.read_json("lånekassen_data.json")
lånekassen_data

,doc_hash,lang,url,domain,date,mimetype,fulltext
0,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,nno,http://lanekassen.no/globalassets/brosjyrer-fe...,lanekassen.no,2022-12-19 01:53:28,pdf,[Er du flyktning? Du blir rekna som flyktning ...
1,1fbe096fdfdd9916be1313006ac5cf44ac650231,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:14,pdf,"[, , Du kan bruke dette skjemaet dersom du tar..."
2,582aaf9a510b3bde21b5b3e13ffc3453d6ca8174,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:21,pdf,"[, , Det er viktig at du les informasjonen på ..."
3,32e23cf05e4db850a9e2f622c2edd0482572677e,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:00:45,pdf,[Nynorsk Skjema for lærlinglønn Kor stort bort...
4,4ea46648c7ce59fe0c39b45779ed6402d8b41369,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:01:01,pdf,[Nynorsk Skjema I – artikkelnr. 9001550 – nyno...
...,...,...,...,...,...,...,...
561,e29a43da7165d92d937c56416d50f563201ab5fa,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:18,pdf,"[, , 01.01.2020 Storebrand Bank ASA 70 % Bolig..."
562,c120ee7f582c758ae392b8f90795a7c664c39e90,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:23,pdf,"[, , 06.11.2019 Storebrand Bank ASA 70 % Bolig..."
563,553101a9b0a9fac935cf31e5dd59555d23ecbefe,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:41,pdf,"[, , Flyktningstipendet Mottakere av flyktning..."
564,1f14c7bcc564613f79be7a8cae1a946e27cfd755,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:44,pdf,"[, , Tall og fakta om Lånekassens kunder og or..."


In [8]:
len(set(lånekassen_data.url)), len(set(lånekassen_data.doc_hash))

(566, 566)

In [9]:
# Max antall mulige setningspar
nob_antall_setninger = lånekassen_data[lånekassen_data.lang == "nob"].fulltext.apply(len).sum()
nno_antall_setninger = lånekassen_data[lånekassen_data.lang == "nno"].fulltext.apply(len).sum()
min((nno_antall_setninger, nob_antall_setninger))

5958

In [10]:
url_to_doc_hash = {e.url: e.doc_hash for e in lånekassen_data.itertuples()}

In [11]:
df["nb_doc_hash"] = df.nb_urls.apply(lambda x: [url_to_doc_hash[url] for url in x if url in url_to_doc_hash])
df["nn_doc_hash"] = df.nn_urls.apply(lambda x: [url_to_doc_hash[url] for url in x if url in url_to_doc_hash])

In [18]:
har_nn_hash = df[df.nn_doc_hash.apply(len)>=1]
har_nb_hash = df[df.nb_doc_hash.apply(len)>=1]
har_hash = df[(df.nn_doc_hash.apply(len)>=1) & (df.nb_doc_hash.apply(len)>=1)]

len(har_nb_hash), len(har_nn_hash), len(har_hash)

(1837, 1865, 1835)

In [19]:
print(f"""
Av {len(df)} setningspar er det 
{len(har_nb_hash)} bokmålsetninger med url som mapper til en hash (og {len(df)-len(har_nb_hash)} som mangler)
{len(har_nn_hash)} nynorsksetninger med url som mapper til en hash (og {len(df)-len(har_nn_hash)} som mangler)
{len(har_hash)} setningspar med url som mapper til hash for begge språk (og {len(df)-len(har_hash)} som mangler)
""")


Av 2278 setningspar er det 
1837 bokmålsetninger med url som mapper til en hash (og 441 som mangler)
1865 nynorsksetninger med url som mapper til en hash (og 413 som mangler)
1835 setningspar med url som mapper til hash for begge språk (og 443 som mangler)



# Sammenlikn med våre matches

In [44]:
diff_pairs = df_[df_.bm_text != df_.nb_segment]
diff_pairs[["nn_text", "bm_text", "nb_segment"]]

,nn_text,bm_text,nb_segment
14,Av omsyn til personvernet ditt skal du ikkje s...,"Av hensyn til personvernet ditt, skal du ikke ...",Av hensyn til personvernet ditt skal du ikke s...
50,"Dersom du har fått etterbetalingar, kan vi sjå...","Dersom du har fått etterbetalinger, kan vi se ...","Dersom du har fått etterbetalinger, kan det se..."
63,Dersom du vil betale inn ekstra kan du gjere d...,Hvis du vil betale inn ekstra kan du gjøre det...,Hvis du vil betale inn ekstra kan du gjøre det...
94,Dette gjeld også om du har brukt opp dei 36 be...,Dette gjelder også om du har brukt opp de 36 b...,Dette gjelder også om du har brukt opp de 36 b...
119,Du kan få stipend og lån for å ta yrkesutdanni...,Du kan få stipend og lån for å gå på videregåe...,Du kan få stipend og lån til å ta yrkesutdanni...
128,Du kan søke om inntil,Du kan søke om inntil,Kan du søke om inntil
136,Du må også sende inn dokumentasjon på kor mykj...,Du må også sende dokumentasjon på hvor mye du ...,Du må også sende inn dokumentasjon på hvor mye...
146,Du skal berre melde inn endring av studiebelas...,Du skal kun melde inn endring av studiebelastn...,Du skal kun melde inn endring av studiebelastn...
306,"Når du er ferdig med utdanninga di, om du går ...","Hvis du er ferdig med utdanningen din, skal st...","Når du er ferdig med utdanningen din, hvis du ..."
321,Omgjeringa skjer først året etter du er ferdig.,Omgjøringen skjer først året etter at du er fe...,Omgjøringen skjer først året etter du er ferdig.


In [65]:
from pathlib import Path
import pandas as pd

p = Path("output")


for e_ in p.iterdir():
    if e_.is_dir():
        for e in e_.iterdir():
            if "flat" in e.name:
                df_ = pd.read_csv(e)
                antall_par_oss = len(df_)
                df_ = df_.merge(df, left_on="nn_text", right_on="nn_segment")
                same_pairs = df_[df_.bm_text == df_.nb_segment]

                print(f"Med {e_.name}/{e.name} fant vi {antall_par_oss} par (mot {len(df)} i omsetjingsminne) \
                \nAv disse var det {len(df_)} av våre nynorsktekster som fantes ordlikt i omsetjingsminnet ({round(len(df_)/antall_par_oss*100, 2)}% av våre par og {round(len(df_)/len(df)*100, 2)}% av para i omsetjingsminnet) \
                \nAv de {len(df_)} ordlike nynorsktekstene, hadde vi samme bokmålstekst som omsetjingsminnet i {len(same_pairs)} tilfeller \
                \nDette er {round(len(same_pairs)/len(df_)*100, 2)}% av parene\n\n")


Med nb_sbert_extend/texts_flat.csv fant vi 2701 par (mot 2278 i omsetjingsminne)                 
Av disse var det 438 av våre nynorsktekster som fantes ordlikt i omsetjingsminnet (16.22% av våre par og 19.23% av para i omsetjingsminnet)                 
Av de 438 ordlike nynorsktekstene, hadde vi samme bokmålstekst som omsetjingsminnet i 427 tilfeller                 
Dette er 97.49% av parene


Med nb_sbert_extend/maxlen_parts_flat_max_pooling.csv fant vi 2697 par (mot 2278 i omsetjingsminne)                 
Av disse var det 438 av våre nynorsktekster som fantes ordlikt i omsetjingsminnet (16.24% av våre par og 19.23% av para i omsetjingsminnet)                 
Av de 438 ordlike nynorsktekstene, hadde vi samme bokmålstekst som omsetjingsminnet i 427 tilfeller                 
Dette er 97.49% av parene


Med nb_sbert_extend/maxlen_parts_flat_mean_pooling.csv fant vi 2700 par (mot 2278 i omsetjingsminne)                 
Av disse var det 438 av våre nynorsktekster som fantes ordlikt 